In [ ]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)
random.seed(42)

n_rows = 1480

departments = ['HR', 'Engineering', 'Sales', 'Marketing', 'Finance']
cities = ['New York', 'San Francisco', 'Chicago', 'Boston', 'Austin']
genders = ['Female', 'Male', 'Non-binary', None]

data = {
    'Employee_ID': [f"EMP_{1000 + i}" for i in range(n_rows)],
    'Name': [f"Employee_{i}" for i in range(n_rows)],
    'Age': np.random.choice([np.nan, 22, 25, 30, 35, 40, 45, 50, 55, 60, 120], size=n_rows, p=[0.05, 0.1, 0.15, 0.15, 0.15, 0.1, 0.1, 0.1, 0.05, 0.03, 0.02]),
    'Department': np.random.choice(departments, size=n_rows),
    'City': np.random.choice(cities, size=n_rows),
    'Gender': np.random.choice(genders, size=n_rows, p=[0.45, 0.45, 0.05, 0.05]),
    'Salary': np.random.choice([None, '45000', '60000', '75000', '85000', '120000', '150000', '999999'], size=n_rows, p=[0.05, 0.2, 0.2, 0.2, 0.15, 0.1, 0.08, 0.02]),
    'Join_Date': [random.choice(['2020-01-15', '2021-05-20', '2019/11/12', '2022-08-01', 'InvalidDate', None]) for _ in range(n_rows)]
}

df_raw = pd.DataFrame(data)

duplicates = df_raw.sample(10, random_state=42)
df_raw = pd.concat([df_raw, duplicates], ignore_index=True)

df_raw.to_csv('raw_employee_data.csv', index=False)
print(f"Raw dataset created with shape: {df_raw.shape}")

Raw dataset created with shape: (1490, 8)


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('raw_employee_data.csv')

initial_rows = len(df)
initial_missing = df.isnull().sum().sum()
initial_duplicates = df.duplicated().sum()

df = df.drop_duplicates()

df['Department'] = df['Department'].str.strip().str.title()
df['City'] = df['City'].str.strip().str.title()
df['Gender'] = df['Gender'].str.strip().str.title()

df['Gender'] = df['Gender'].fillna('Unknown')

df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')
df['Salary'] = df.groupby('Department')['Salary'].transform(lambda x: x.fillna(x.median()))

df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

Q1_age, Q3_age = df['Age'].quantile(0.25), df['Age'].quantile(0.75)
IQR_age = Q3_age - Q1_age
median_age = df['Age'].median()
df.loc[(df['Age'] > 65) | (df['Age'] < 18), 'Age'] = median_age
df['Age'] = df['Age'].fillna(median_age)

Q1_sal, Q3_sal = df['Salary'].quantile(0.25), df['Salary'].quantile(0.75)
IQR_sal = Q3_sal - Q1_sal
median_sal = df['Salary'].median()
df.loc[df['Salary'] > 200000, 'Salary'] = median_sal
df['Salary'] = df['Salary'].fillna(median_sal)

df['Join_Date'] = pd.to_datetime(df['Join_Date'], errors='coerce')
df['Join_Date'] = df['Join_Date'].ffill().bfill()

final_rows = len(df)
final_missing = df.isnull().sum().sum()
final_duplicates = df.duplicated().sum()

summary_df = pd.DataFrame({
    'Metric': ['Total Rows', 'Missing Values', 'Duplicate Rows'],
    'Before': [initial_rows, initial_missing, initial_duplicates],
    'After': [final_rows, final_missing, final_duplicates]
})

df.to_csv('clean_data.csv', index=False)
print("=== Task 1: Data Cleaning Summary ===")
print(summary_df)

=== Task 1: Data Cleaning Summary ===
           Metric  Before  After
0      Total Rows    1490   1480
1  Missing Values     491      0
2  Duplicate Rows      10      0


In [ ]:
print("=== Numeric Columns Summary Statistics ===")
print(df[['Age', 'Salary']].describe())

print("\n=== Group Aggregation (Department Profile) ===")
dept_profile = df.groupby('Department').agg(
    Average_Salary=('Salary', 'mean'),
    Median_Age=('Age', 'median'),
    Employee_Count=('Employee_ID', 'count')
).reset_index()
print(dept_profile)

print("\n=== Correlation Matrix ===")
corr_matrix = df[['Age', 'Salary']].corr()
print(corr_matrix)

print("\n=== Business Questions & Answers ===")
max_sal_city = df.groupby('City')['Salary'].mean().idxmax()
max_sal_city_val = df.groupby('City')['Salary'].mean().max()
print(f"Q1: Which city has the highest average salary? -> {max_sal_city} (${max_sal_city_val:,.2f})")

gender_dist = df['Gender'].value_counts(normalize=True) * 100
print(f"Q2: What is the company gender distribution? -> {gender_dist.to_dict()}")

avg_age_eng = df[df['Department'] == 'Engineering']['Age'].mean()
print(f"Q3: What is the average age in Engineering? -> {avg_age_eng:.1f} years")

budget_data = {
    'Department': ['HR', 'Engineering', 'Sales', 'Marketing', 'Finance'],
    'Budget_M': [1.5, 5.0, 3.5, 2.0, 2.5]
}
df_budget = pd.DataFrame(budget_data)

df_merged = pd.merge(df, df_budget, on='Department', how='left')
print("\n=== Merged Dataset Preview (with Budgets) ===")
print(df_merged.head())

=== Numeric Columns Summary Statistics ===
               Age         Salary
count  1480.000000    1480.000000
mean     36.295270   77341.216216
std      10.483767   29092.804408
min      22.000000   45000.000000
25%      25.000000   60000.000000
50%      35.000000   75000.000000
75%      45.000000   85000.000000
max      60.000000  150000.000000

=== Group Aggregation (Department Profile) ===
    Department  Average_Salary  Median_Age  Employee_Count
0  Engineering    77709.090909        35.0             275
1      Finance    79493.464052        35.0             306
2           Hr    75290.102389        35.0             293
3    Marketing    77601.880878        35.0             319
4        Sales    76498.257840        35.0             287

=== Correlation Matrix ===
             Age    Salary
Age     1.000000  0.016098
Salary  0.016098  1.000000

=== Business Questions & Answers ===
Q1: Which city has the highest average salary? -> Boston ($79,407.67)
Q2: What is the company gender d